In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import string
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM, Dense

I0000 00:00:1789582092.815612    4646 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv("quotes.csv")


In [3]:
df.head()

,quote,author,category
0,"I'm selfish, impatient and a little insecure. ...",Marilyn Monroe,"attributed-no-source, best, life, love, mistak..."
1,You've gotta dance like there's nobody watchin...,William W. Purkey,"dance, heaven, hurt, inspirational, life, love..."
2,You know you're in love when you can't fall as...,Dr. Seuss,"attributed-no-source, dreams, love, reality, s..."
3,A friend is someone who knows all about you an...,Elbert Hubbard,"friend, friendship, knowledge, love"
4,Darkness cannot drive out darkness: only light...,"Martin Luther King Jr., A Testament of Hope: T...","darkness, drive-out, hate, inspirational, ligh..."


In [4]:
df.shape

(499709, 3)

In [5]:

print(df.shape)
print(df.isnull().sum())

(499709, 3)
quote          1
author      1753
category      63
dtype: int64


In [6]:
null_percentage = (df.isnull().sum() / len(df)) * 100
print(null_percentage)

quote       0.000200
author      0.350804
category    0.012607
dtype: float64


In [7]:
print("Empty rows:", df.isnull().all(axis=1).sum())

Empty rows: 1


In [8]:
print("Empty quotes:", (df["quote"].fillna("").str.strip() == "").sum())
print("Empty authors:", (df["author"].fillna("").str.strip() == "").sum())
print("Empty categories:", (df["category"].fillna("").str.strip() == "").sum())

Empty quotes: 1
Empty authors: 1754
Empty categories: 63


In [9]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate quotes:", df["quote"].duplicated().sum())

Duplicate rows: 2442
Duplicate quotes: 5919


In [10]:
# Remove rows where quote is null or empty
df = df[df["quote"].notna()]
df = df[df["quote"].str.strip() != ""]

print(df.shape)

(499708, 3)


In [11]:
df = df.drop_duplicates(subset=["quote"])

print(df.shape)

(493789, 3)


In [12]:
df = df.reset_index(drop=True)

print(df.shape)

(493789, 3)


In [13]:
print(df.isnull().sum())
print("Duplicate quotes:", df["quote"].duplicated().sum())

quote          0
author      1752
category      61
dtype: int64
Duplicate quotes: 0


In [14]:
df.shape

(493789, 3)

In [15]:
print(df.shape)

print("\nNull values:")
print(df.isnull().sum())

print("\nEmpty quotes:")
print((df["quote"].fillna("").str.strip() == "").sum())

print("\nDuplicate quotes:")
print(df["quote"].duplicated().sum())

(493789, 3)

Null values:
quote          0
author      1752
category      61
dtype: int64

Empty quotes:
0

Duplicate quotes:
0


In [16]:
df["quote_length"] = df["quote"].str.len()

print(df["quote_length"].describe())

count    493789.000000
mean        200.020977
std         236.137639
min           1.000000
25%          75.000000
50%         129.000000
75%         235.000000
max        3999.000000
Name: quote_length, dtype: float64


In [17]:
print("Shortest quote:")
print(df.loc[df["quote_length"].idxmin(), "quote"])

print("\nLongest quote:")
print(df.loc[df["quote_length"].idxmax(), "quote"])

Shortest quote:
1

Longest quote:
I imagined my coffin being closed, and the screws being turned. I was immobile, but I was alive, and I wanted to tell my family that I was seeing everything. I wanted to tell them all that I loved them, but not a sound came out of my mouth. My father and mother were weeping, my wife and my friends were gathered around, but I was completely alone! With all of the people dear to me standing there, no one was able to see that I was alive and that I had not yet accomplished all that I wanted to do in this world. I tried desperately to open my eyes, to give a sign, to beat on the lid of the coffin. But I could not move any part of my body. I felt the coffin being carried toward the grave. I could hear the sound of the handles grinding against their fittings, the steps of those in the procession, and conversations from this side and that. Someone said that he had a date for dinner later on, and another observed that I had died early. The smell of flowers all

In [18]:
print("\nQuotes <= 50 chars:", (df["quote_length"] <= 50).sum())
print("Quotes <= 100 chars:", (df["quote_length"] <= 100).sum())
print("Quotes <= 200 chars:", (df["quote_length"] <= 200).sum())
print("Quotes <= 300 chars:", (df["quote_length"] <= 300).sum())


Quotes <= 50 chars: 52326
Quotes <= 100 chars: 188727
Quotes <= 200 chars: 340989
Quotes <= 300 chars: 410374


In [19]:
all_text = "".join(df["quote"].astype(str))

chars = sorted(set(all_text))

print("Total characters:", len(all_text))
print("Unique characters:", len(chars))
print(chars)

Total characters: 98768158
Unique characters: 1136
['\x03', '\t', '\n', '\x0b', '\x0c', '\x0f', '\x19', '\x1b', '\x1c', '\x1d', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '\x7f', '\x92', '\x93', '\x94', '\x97', '\x9d', '\xa0', '¡', '¢', '£', '¥', '¦', '§', '¨', '©', '«', '¬', '\xad', '®', '¯', '°', '´', '¶', '·', '¸', '»', '½', '¾', '¿', 'À', 'Á', 'Â', 'Ã', 'Ä', 'Æ', 'Ç', 'È', 'É', 'Ê', 'Í', 'Î', 'Ï', 'Ò', 'Ó', 'Ô', 'Õ', 'Ö', '×', 'Ú', 'Ü', 'Ý', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ò', 'ó', 'ô', 'õ', 'ö

In [20]:
from collections import Counter

char_counts = Counter(all_text)

print(char_counts.most_common(30))

[(' ', 17376537), ('e', 9661315), ('t', 6989181), ('o', 6250067), ('a', 5879441), ('n', 5404130), ('i', 5256580), ('s', 4867674), ('r', 4377009), ('h', 4112640), ('l', 3234229), ('d', 2818839), ('u', 2417336), ('m', 1899859), ('c', 1855962), ('y', 1800753), ('f', 1678721), ('w', 1644516), ('g', 1594942), ('p', 1320951), ('.', 1200043), ('b', 1112814), (',', 999735), ('v', 899667), ('k', 649286), ('I', 491386), ("'", 319806), ('T', 241069), ('A', 149562), ('W', 131554)]


In [21]:
import re

# Characters that are NOT basic English letters, numbers,
# common punctuation and whitespace
allowed_pattern = r"[^a-zA-Z0-9\s.,!?;:'\"()\-—–…&%$@#]"

unusual_counts = df["quote"].str.count(allowed_pattern).sum()

print("Unusual character occurrences:", unusual_counts)

Unusual character occurrences: 238355


In [22]:
mask = df["quote"].str.contains(
    allowed_pattern,
    regex=True,
    na=False
)

print(df.loc[mask, "quote"].head(20).to_string(index=False))

Only once in your life, I truly believe, you fi...
We’re all a little weird. And life is a little ...
He’s not perfect. You aren’t either, and the tw...
And now I’m looking at you,” he said, “and you’...
I think you still love me, but we can’t escape ...
I heard what you said. I’m not the silly romant...
Love is always patient and kind. It is never je...
They didn’t agree on much. In fact, they didn’t...
If he’s not calling you, it’s because you are n...
We’re all seeking that special person who is ri...
My dear,Find what you love and let it kill you....
It’s probably not just by chance that I’m alone...
He does something to me, that boy. Every time. ...
Before you, Bella, my life was like a moonless ...
              Happiness [is] only real when shared
Art and love are the same thing: It’s the proce...
Peeta, how come I never know when you're having...
I guess that’s just part of loving people: You ...
People have forgotten this truth," the fox said...
Stop fighting me!" he said, try

In [ ]:
import unicodedata
def clean_text(text):
    text = unicodedata.normalize("NFKC", str(text))

    # Remove control characters except newline and tab
    text = "".join(
        char for char in text
        if unicodedata.category(char) != "Cc"
        or char in "\n\t"
    )

    return text

In [25]:
df["quote"] = df["quote"].apply(clean_text)

In [26]:
all_text = "".join(df["quote"])

chars = sorted(set(all_text))

print("Total characters:", len(all_text))
print("Unique characters:", len(chars))
print(chars)

Total characters: 98785338
Unique characters: 1089
['\t', '\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '¡', '¢', '£', '¥', '¦', '§', '©', '«', '¬', '\xad', '®', '°', '¶', '·', '»', '¿', 'À', 'Á', 'Â', 'Ã', 'Ä', 'Æ', 'Ç', 'È', 'É', 'Ê', 'Í', 'Î', 'Ï', 'Ò', 'Ó', 'Ô', 'Õ', 'Ö', '×', 'Ú', 'Ü', 'Ý', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ò', 'ó', 'ô', 'õ', 'ö', '÷', 'ø', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'Ā', 'ā', 'ă', 'ą', 'ć', 'č', 'Đ', 'đ', 'Ē', 'ē', 'ė', 'ę', 'ě', 'ğ', 'ġ', 'ħ', 'ĩ', 'ī', 'İ', 'ı', 'ł', 'ń

In [27]:
import re

# English-oriented character set
allowed_chars = set(
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    " .,!?;:'\"()-"
)

def count_non_english_chars(text):
    return sum(char not in allowed_chars for char in text)

non_english_counts = df["quote"].apply(count_non_english_chars)

print("Quotes containing non-English/special characters:",
      (non_english_counts > 0).sum())

print("Total non-English/special character occurrences:",
      non_english_counts.sum())

print("Percentage of quotes affected:",
      (non_english_counts > 0).mean() * 100)

Quotes containing non-English/special characters: 72539
Total non-English/special character occurrences: 270576
Percentage of quotes affected: 14.690282691594994


In [28]:
import unicodedata

def normalize_punctuation(text):
    replacements = {
        "’": "'",
        "‘": "'",
        "“": '"',
        "”": '"',
        "–": "-",
        "—": "-",
        "…": "...",
        "−": "-",
        "\u00a0": " ",   # non-breaking space
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text

df["quote"] = df["quote"].apply(normalize_punctuation)

In [29]:
allowed_chars = set(
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    " .,!?;:'\"()-"
)

def is_valid_quote(text):
    return all(char in allowed_chars for char in text)

df = df[df["quote"].apply(is_valid_quote)].copy()

df = df.reset_index(drop=True)

print(df.shape)

(477626, 4)


In [30]:
all_text = "".join(df["quote"])

chars = sorted(set(all_text))

print("Total characters:", len(all_text))
print("Unique characters:", len(chars))
print(chars)

Total characters: 92563037
Unique characters: 74
[' ', '!', '"', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [32]:
df.head()

,quote,author,category,quote_length
0,"I'm selfish, impatient and a little insecure. ...",Marilyn Monroe,"attributed-no-source, best, life, love, mistak...",202
1,You've gotta dance like there's nobody watchin...,William W. Purkey,"dance, heaven, hurt, inspirational, life, love...",149
2,You know you're in love when you can't fall as...,Dr. Seuss,"attributed-no-source, dreams, love, reality, s...",102
3,A friend is someone who knows all about you an...,Elbert Hubbard,"friend, friendship, knowledge, love",64
4,Darkness cannot drive out darkness: only light...,"Martin Luther King Jr., A Testament of Hope: T...","darkness, drive-out, hate, inspirational, ligh...",110


In [33]:
df = df.drop(columns=["quote_length"])

print(df.shape)

(477626, 3)


In [34]:
# Remove rows where category is missing or empty
df = df[df["category"].notna()]
df = df[df["category"].str.strip() != ""]

df = df.reset_index(drop=True)

print(df.shape)
print(df["category"].isnull().sum())

(477565, 3)
0


In [ ]:

print("\nNull values:")
print(df.isnull().sum())

print("\nEmpty quotes:")
print((df["quote"].fillna("").str.strip() == "").sum())

print("\nDuplicate quotes:")
print(df["quote"].duplicated().sum())

(477565, 3)

Null values:
quote          0
author      1741
category       0
dtype: int64

Empty quotes:
0

Duplicate quotes:
216


In [36]:
# Remove duplicate quotes
df = df.drop_duplicates(subset=["quote"]).reset_index(drop=True)

print(df.shape)
print("Duplicate quotes:", df["quote"].duplicated().sum())

(477349, 3)
Duplicate quotes: 0


In [37]:
print("Number of unique categories:", df["category"].nunique())

print("\nTop 20 categories:")
print(df["category"].value_counts().head(20))

print("\nSample categories:")
print(df["category"].head(10).tolist())

Number of unique categories: 355494

Top 20 categories:
category
education, happiness, hope, inspirational, intelligence, knowledge, life, love, philosophy, quotes, truth, wisdom    1648
happiness                                                                                                             870
friendship                                                                                                            829
prayer                                                                                                                819
kindlehighlight                                                                                                       742
inspirational                                                                                                         737
writing                                                                                                               701
love                                                                             

In [38]:
# Split category strings into individual categories
df["category_list"] = df["category"].str.split(",")

# Remove extra spaces
df["category_list"] = df["category_list"].apply(
    lambda x: [cat.strip() for cat in x]
)

# Get all individual categories
all_categories = df["category_list"].explode()

print("Unique individual categories:", all_categories.nunique())

print("\nTop 20 individual categories:")
print(all_categories.value_counts().head(20))

Unique individual categories: 145917

Top 20 individual categories:
category_list
love                    37512
life                    34111
inspirational           28365
philosophy              14531
humor                   13471
god                     12161
truth                   11448
wisdom                  10507
happiness               10015
hope                     9395
inspirational-quotes     9021
quotes                   9008
romance                  8709
faith                    8668
death                    7958
inspiration              7903
success                  7688
writing                  7621
poetry                   6791
religion                 6784
Name: count, dtype: int64


In [39]:
# Get the top 50 most frequent categories
top_categories = all_categories.value_counts().head(50)

print("Top 50 categories:")
print(top_categories)

Top 50 categories:
category_list
love                    37512
life                    34111
inspirational           28365
philosophy              14531
humor                   13471
god                     12161
truth                   11448
wisdom                  10507
happiness               10015
hope                     9395
inspirational-quotes     9021
quotes                   9008
romance                  8709
faith                    8668
death                    7958
inspiration              7903
success                  7688
writing                  7621
poetry                   6791
religion                 6784
knowledge                6252
education                6143
motivational             5954
time                     5829
spirituality             5499
relationships            5495
Life                     5371
You                      5348
life-lessons             5209
motivation               5173
fear                     5164
People                   5074
books  

In [40]:
# Keep only quotes that have at least one of the top 50 categories
top_50 = set(top_categories.index)

df["category"] = df["category_list"].apply(
    lambda cats: [cat for cat in cats if cat in top_50]
)

# Remove rows with no top-50 category
df = df[df["category"].apply(len) > 0].reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Sample categories:", df["category"].head().tolist())

Dataset shape: (221430, 4)
Sample categories: [['life', 'love', 'truth'], ['inspirational', 'life', 'love'], ['dreams', 'love'], ['friendship', 'knowledge', 'love'], ['inspirational', 'love', 'peace']]


In [41]:
# Prepare quote-level data before train-validation split
quote_df = df[["quote", "category"]].copy()

print("Quote-level dataset shape:", quote_df.shape)
print(quote_df.head())

Quote-level dataset shape: (221430, 2)
                                               quote  \
0  I'm selfish, impatient and a little insecure. ...   
1  You've gotta dance like there's nobody watchin...   
2  You know you're in love when you can't fall as...   
3  A friend is someone who knows all about you an...   
4  Darkness cannot drive out darkness: only light...   

                        category  
0            [life, love, truth]  
1    [inspirational, life, love]  
2                 [dreams, love]  
3  [friendship, knowledge, love]  
4   [inspirational, love, peace]  


In [42]:
from sklearn.model_selection import train_test_split

# Split at QUOTE level first to prevent data leakage
train_quotes, val_quotes = train_test_split(
    quote_df,
    test_size=0.1,
    random_state=42
)

print("Training quotes:", len(train_quotes))
print("Validation quotes:", len(val_quotes))

Training quotes: 199287
Validation quotes: 22143


In [43]:
# Explode categories AFTER the train-validation split

train_df = train_quotes.explode("category").reset_index(drop=True)
val_df = val_quotes.explode("category").reset_index(drop=True)

# Add special category markers
train_df["text"] = (
    "<CAT> " + train_df["category"].astype(str)
    + " <START> " + train_df["quote"].astype(str)
)

val_df["text"] = (
    "<CAT> " + val_df["category"].astype(str)
    + " <START> " + val_df["quote"].astype(str)
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

print("\nFirst training example:")
print(train_df["text"].iloc[0])

Training samples: 369134
Validation samples: 40649

First training example:
<CAT> faith <START> The purest form of faith happens when you reach the bottom of your reasoning and find there is nothing that you can do that will make sense out of what you have been through.


In [44]:
def generate_quote(
    category,
    max_words=30,
    temperature=0.7,
    repetition_penalty=1.5,
    min_words=8
):
    sequence = tokenizer.texts_to_sequences(
        [f"<CAT> {category} <START>"]
    )[0]

    generated_tokens = []

    for _ in range(max_words):

        input_sequence = sequence[-(MAX_LEN - 1):]

        padded = pad_sequences(
            [input_sequence],
            maxlen=MAX_LEN - 1,
            padding="post"
        )

        predictions = model.predict(padded, verbose=0)

        last_position = len(input_sequence) - 1
        probabilities = predictions[0, last_position].astype("float64")

        # Never generate OOV
        oov_id = tokenizer.word_index.get("<OOV>")
        if oov_id is not None and oov_id < len(probabilities):
            probabilities[oov_id] = 0

        # Reduce repetition
        for token in set(generated_tokens[-5:]):
            if token < len(probabilities):
                probabilities[token] /= repetition_penalty

        # Temperature
        probabilities = np.log(probabilities + 1e-8) / temperature
        probabilities = np.exp(probabilities)
        probabilities /= probabilities.sum()

        next_token = np.random.choice(
            len(probabilities),
            p=probabilities
        )

        next_word = tokenizer.index_word.get(next_token, "")

        # Stop at special tokens
        if next_token == 0 or next_word.lower() in [
            "<cat>", "<start>", "<oov>"
        ]:
            break

        sequence.append(next_token)
        generated_tokens.append(next_token)

        # Stop after natural sentence ending
        generated_words = [
            tokenizer.index_word.get(t, "")
            for t in generated_tokens
        ]

        if len(generated_words) >= min_words:
            if next_word.endswith((".", "!", "?")):
                break

    words = []

    for token in sequence:
        word = tokenizer.index_word.get(token, "")

        if word and word.lower() not in [
            "<cat>", "<start>", "<oov>"
        ]:
            words.append(word)

    return " ".join(words)

In [45]:
import re

def fix_punctuation_spacing(text):
    text = str(text)

    # Add a space after punctuation when directly followed by a letter
    text = re.sub(r'([,.!?;:])([A-Za-z])', r'\1 \2', text)

    # Clean multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# Apply punctuation cleanup to the quote-level dataset
quote_df["quote"] = quote_df["quote"].apply(
    fix_punctuation_spacing
)

print("Punctuation cleanup completed.")

Punctuation cleanup completed.


In [49]:
from tensorflow.keras.preprocessing.text import Tokenizer

VOCAB_SIZE = 40000

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>",
    filters="",
    lower=False
)

# Fit tokenizer ONLY on training data
tokenizer.fit_on_texts(train_df["text"])

print("Total vocabulary:", len(tokenizer.word_index))
print("Usable vocabulary:", VOCAB_SIZE)

print("CAT ID:", tokenizer.word_index.get("<CAT>"))
print("START ID:", tokenizer.word_index.get("<START>"))
print("OOV ID:", tokenizer.word_index.get("<OOV>"))

Total vocabulary: 282583
Usable vocabulary: 40000
CAT ID: 3
START ID: 4
OOV ID: 1


In [50]:
# Convert training and validation text into token sequences

train_sequences = tokenizer.texts_to_sequences(
    train_df["text"]
)

val_sequences = tokenizer.texts_to_sequences(
    val_df["text"]
)

print("Training sequences:", len(train_sequences))
print("Validation sequences:", len(val_sequences))

print("\nFirst training sequence:")
print(train_sequences[0][:20])

Training sequences: 369134
Validation sequences: 40649

First training sequence:
[3, 128, 4, 29, 4967, 547, 7, 128, 660, 41, 10, 567, 2, 2335, 7, 14, 4611, 6, 124, 72]


In [51]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 60

# Input  = all tokens except the last
# Target = all tokens except the first

X_train = [seq[:-1] for seq in train_sequences]
y_train = [seq[1:] for seq in train_sequences]

X_val = [seq[:-1] for seq in val_sequences]
y_val = [seq[1:] for seq in val_sequences]

# Post-padding
X_train = pad_sequences(
    X_train,
    maxlen=MAX_LEN - 1,
    padding="post",
    truncating="post"
)

y_train = pad_sequences(
    y_train,
    maxlen=MAX_LEN - 1,
    padding="post",
    truncating="post"
)

X_val = pad_sequences(
    X_val,
    maxlen=MAX_LEN - 1,
    padding="post",
    truncating="post"
)

y_val = pad_sequences(
    y_val,
    maxlen=MAX_LEN - 1,
    padding="post",
    truncating="post"
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nPadding percentage:")
print("X_train:", np.mean(X_train == 0) * 100, "%")
print("y_train:", np.mean(y_train == 0) * 100, "%")
print("X_val:", np.mean(X_val == 0) * 100, "%")
print("y_val:", np.mean(y_val == 0) * 100, "%")

X_train: (369134, 59)
y_train: (369134, 59)
X_val: (40649, 59)
y_val: (40649, 59)

Padding percentage:
X_train: 52.494826875142394 %
y_train: 52.494826875142394 %
X_val: 51.94448880473638 %
y_val: 51.94448880473638 %


In [52]:
print(X_train[0])
print(y_train[0])

print("Padding in X:", np.mean(X_train == 0))
print("Padding in y:", np.mean(y_train == 0))

[   3  128    4   29 4967  547    7  128  660   41   10  567    2 2335
    7   14 4611    6  124   72    9  170   13   10   32   48   13   28
   77  343   81    7   35   10   24  110    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0]
[ 128    4   29 4967  547    7  128  660   41   10  567    2 2335    7
   14 4611    6  124   72    9  170   13   10   32   48   13   28   77
  343   81    7   35   10   24  110 2588    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0]
Padding in X: 0.524948268751424
Padding in y: 0.524948268751424


In [53]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense

model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        mask_zero=True
    ),

    LSTM(
        256,
        return_sequences=True
    ),

    Dropout(0.2),

    Dense(
        VOCAB_SIZE,
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.build((None, MAX_LEN - 1))

model.summary()

I0000 00:00:1789582579.991874    4646 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2240 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650 Ti, pci bus id: 0000:01:00.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 59, 128)        │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 59, 256)        │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 59, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 59, 40000)      │    10,280,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,794,240 (60.25 MB)

 Trainable params: 15,794,240 (60.25 MB)

 Non-trainable params: 0 (0.00 B)

In [57]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
11536/11536 ━━━━━━━━━━━━━━━━━━━━ 2943s 255ms/step - accuracy: 0.1830 - loss: 5.3665 - val_accuracy: 0.2087 - val_loss: 4.9984
Epoch 2/10
11536/11536 ━━━━━━━━━━━━━━━━━━━━ 3247s 281ms/step - accuracy: 0.2257 - loss: 4.7226 - val_accuracy: 0.2183 - val_loss: 4.8717
Epoch 3/10
11536/11536 ━━━━━━━━━━━━━━━━━━━━ 3284s 285ms/step - accuracy: 0.2473 - loss: 4.4590 - val_accuracy: 0.2218 - val_loss: 4.8519
Epoch 4/10
11536/11536 ━━━━━━━━━━━━━━━━━━━━ 3274s 284ms/step - accuracy: 0.2636 - loss: 4.2893 - val_accuracy: 0.2227 - val_loss: 4.8692
Epoch 5/10
11536/11536 ━━━━━━━━━━━━━━━━━━━━ 3280s 284ms/step - accuracy: 0.2762 - loss: 4.1667 - val_accuracy: 0.2223 - val_loss: 4.8996
Epoch 6/10
11536/11536 ━━━━━━━━━━━━━━━━━━━━ 3292s 285ms/step - accuracy: 0.2860 - loss: 4.0732 - val_accuracy: 0.2212 - val_loss: 4.9335
Epoch 7/10
11536/11536 ━━━━━━━━━━━━━━━━━━━━ 3298s 286ms/step - accuracy: 0.2940 - loss: 4.0008 - val_accuracy: 0.2211 - val_loss: 4.9636
Epoch 8/10
11536/11536 ━━━━━━━━━━━━━━━━━━

In [58]:
print(generate_quote("love"))
print(generate_quote("life"))
print(generate_quote("wisdom"))
print(generate_quote("happiness"))

love I think it was a whole lot of people who had to see a business in the last day.
life It is joy that makes us our ultimate purpose of life and not to find fault on to be the ultimate purpose of our life we are never going to
wisdom The greatest violence, and the greatest threat to humanity, is the growth of MONEY.
happiness The most beautiful person in this world is not in the best way to learn.


In [59]:
model.save("quote_lstm_model_v3.keras")

In [60]:
import pickle

with open("tokenizer_v3.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [61]:
with open("history_v3.pkl", "wb") as f:
    pickle.dump(history.history, f)